In [1]:
# -*- coding: utf-8 -*-
"""
Version améliorée:
 - Ajout du contexte temporel (t-2, t-1, t+1, t+2)
 - Nouvelles features: Hjorth params, RMS, spectral entropy, kurtosis, skewness
 - LGBM hyperparams optimisés + class_weight="balanced"
"""

import os
from pathlib import Path
import json
import joblib
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

# plotting (optionnel)
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(style='whitegrid', context='talk')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False

# signal & stats
from scipy.signal import welch, butter, filtfilt
from scipy.stats import kurtosis, skew, entropy
from math import log

# sklearn
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import accuracy_score, classification_report, f1_score

# lightgbm
import lightgbm as lgb
from lightgbm import early_stopping, log_evaluation

In [4]:
# ==========================================
# CONFIG
# ==========================================
BUNDLE_DIR = Path('/Users/MC/Desktop/CS 3A/Cours/Machine Learning 🖥️/Kaggle Project/beacon-biosignals-sleep-staging-2025')
MODEL_DIR = Path('/Users/MC/Desktop/CS 3A/Cours/Machine Learning 🖥️/Kaggle Project/beacon-biosignals-sleep-staging-2025/models')
MODEL_DIR.mkdir(parents=True, exist_ok=True)
SUBMISSION_OUT = MODEL_DIR / "lgbm_features_submission.csv"
MODEL_OUT = MODEL_DIR / "lgbm_features.joblib"

TEST_SIZE = 0.25
RANDOM_STATE = 1234

VALID_LABELS = [0, 1, 2, 3, 4]
SAMPLING_RATE = 100
EPOCH_DURATION_SECONDS = 30
SAMPLES_PER_EPOCH = SAMPLING_RATE * EPOCH_DURATION_SECONDS
NUM_CHANNELS = 5

print("DATA DIR :", BUNDLE_DIR)

DATA DIR : /Users/MC/Desktop/CS 3A/Cours/Machine Learning 🖥️/Kaggle Project/beacon-biosignals-sleep-staging-2025


In [5]:
# ==========================================
# DATA LOADING HELPERS (identiques)
# ==========================================
def _discover_record_ids(base_dir: Path, split: str):
    manifest = base_dir / "records_manifest.csv"
    if manifest.exists():
        df = pd.read_csv(manifest)
        return df[df["split"] == split]["record_id"].astype(int).tolist()

    signal_dir = base_dir / f"signals_{split}"
    record_ids = []
    for p in sorted(signal_dir.glob("signal_*.npy")):
        record_ids.append(int(p.stem.split("_")[-1]))
    return record_ids


def _load_signal_array(path: Path):
    x = np.load(path).astype(np.float32)
    if x.ndim != 2 or x.shape[0] != NUM_CHANNELS:
        raise ValueError(f"Bad signal shape {x.shape}")
    return x


def _load_target_array(path: Path):
    y = np.load(path).astype(np.int16)
    if y.ndim == 2 and y.shape[0] == NUM_CHANNELS:
        y = y[0]
    return y


def iter_signals_and_targets(base_dir: Path, split: str):
    ids = _discover_record_ids(base_dir, split)
    signal_dir = base_dir / f"signals_{split}"
    target_dir = base_dir / f"targets_{split}"

    for rid in ids:
        sig = signal_dir / f"signal_{rid}.npy"
        tar = target_dir / f"target_{rid}.npy"
        yield rid, _load_signal_array(sig), _load_target_array(tar)


def iter_signals_only(base_dir: Path, split: str):
    ids = _discover_record_ids(base_dir, split)
    signal_dir = base_dir / f"signals_{split}"
    for rid in ids:
        sig = signal_dir / f"signal_{rid}.npy"
        yield rid, _load_signal_array(sig)

In [14]:
# ==========================================
# FEATURE ENGINEERING
# ==========================================

# --- bandpowers via Welch ---
def bandpower_from_psd(freqs, psd, fmin, fmax):
    mask = (freqs >= fmin) & (freqs <= fmax)
    if not np.any(mask):
        return 0.0
    return float(np.trapz(psd[mask], freqs[mask]))

def compute_band_powers(sig, fs=SAMPLING_RATE):
    # welch returns power spectral density estimate
    freqs, psd = welch(sig, fs=fs, nperseg=min(1024, len(sig)))
    return {
        "delta": bandpower_from_psd(freqs, psd, 0.5, 4),
        "theta": bandpower_from_psd(freqs, psd, 4, 8),
        "alpha": bandpower_from_psd(freqs, psd, 8, 12),
        "beta":  bandpower_from_psd(freqs, psd, 12, 30),
        "gamma": bandpower_from_psd(freqs, psd, 30, 45),
        "total_pow": bandpower_from_psd(freqs, psd, 0.5, 45)
    }

# --- Higuchi fractal dimension (déjà dans ton code) ---
def higuchi_fd(x, kmax=10):
    x = np.asarray(x, dtype=float)
    n = len(x)
    if n < 3:
        return 0.0

    Lk = np.zeros(kmax)
    for k in range(1, kmax + 1):
        Lm = []
        for m in range(k):
            idxs = np.arange(m, n, k)
            if len(idxs) < 2:
                continue
            diffs = np.abs(np.diff(x[idxs]))
            Lm.append(np.sum(diffs) * (n - 1) / (len(idxs) * k))
        Lk[k - 1] = np.mean(Lm) if len(Lm) else 0

    valid = Lk > 0
    if valid.sum() < 2:
        return 0.0

    ln_k = np.log(1 / np.arange(1, kmax + 1)[valid])
    ln_Lk = np.log(Lk[valid])
    slope = np.polyfit(ln_k, ln_Lk, 1)[0]
    return float(slope)

# --- MMD (Maximum-Minimum Distance) ---
def compute_mmd(sig, fs=SAMPLING_RATE, window_size=100):
    n = len(sig)
    dt = 1 / fs
    mmd_sum = 0.0

    for start in range(0, n, window_size):
        win = sig[start:start + window_size]
        if len(win) < 2:
            break

        min_val = float(np.min(win))
        max_val = float(np.max(win))

        idx_min = int(np.argmin(win))
        idx_max = int(np.argmax(win))

        time_dist = abs(idx_max - idx_min) * dt
        amp_dist = abs(max_val - min_val)

        mmd_sum += np.sqrt(time_dist**2 + amp_dist**2)

    return float(mmd_sum)

# --- Zero-crossing rate ---
def zero_crossings(sig):
    return float(np.sum(np.diff(np.sign(sig)) != 0))

# --- Hjorth parameters ---
def hjorth_parameters(x):
    """
    Returns (activity, mobility, complexity)
    activity = variance(x)
    mobility = sqrt(var(dx)/var(x))
    complexity = mobility(dx)/mobility(x)
    """
    x = np.asarray(x, dtype=float)
    if x.size < 3:
        return 0.0, 0.0, 0.0
    dx = np.diff(x)
    ddx = np.diff(dx)
    var_x = np.var(x)
    var_dx = np.var(dx)
    var_ddx = np.var(ddx) if ddx.size > 0 else 0.0
    activity = float(var_x)
    mobility = float(np.sqrt(var_dx / (var_x + 1e-12)))
    complexity = float(np.sqrt(var_ddx / (var_dx + 1e-12)) / (mobility + 1e-12))
    return activity, mobility, complexity

# --- RMS ---
def rms(sig):
    return float(np.sqrt(np.mean(sig**2)))

# --- Spectral entropy (normalized) ---
def spectral_entropy(sig, fs=SAMPLING_RATE, nperseg=None):
    freqs, psd = welch(sig, fs=fs, nperseg=nperseg or min(1024, len(sig)))
    psd = np.abs(psd)
    psd = psd / (psd.sum() + 1e-12)
    # Shannon entropy (bits)
    ent = -np.sum(psd * np.log2(psd + 1e-12))
    # normalized
    return float(ent / np.log2(len(psd) + 1e-12))

# --- Neighbor statistics ---
def neighbor_stats(sig):
    if len(sig) < 3:
        return 0.0, 0.0
    prev = sig[:-2]
    curr = sig[1:-1]
    next_ = sig[2:]
    neighbor_mean = np.mean((prev + curr + next_) / 3)
    neighbor_std  = np.mean(np.std(np.vstack([prev, curr, next_]), axis=0))
    return float(neighbor_mean), float(neighbor_std)

# --- Extract features per epoch ---
def extract_epoch_stats(sig):
    """
    sig: numpy array shape (NUM_CHANNELS, total_samples)
    returns DataFrame with one row per epoch with features for each channel
    """
    num_epochs = sig.shape[1] // SAMPLES_PER_EPOCH
    rows = []

    for epoch_idx in range(num_epochs):
        start = epoch_idx * SAMPLES_PER_EPOCH
        end   = start + SAMPLES_PER_EPOCH
        row = {"epoch_index": epoch_idx}

        for ch in range(NUM_CHANNELS):
            ch_sig = sig[ch, start:end].astype(float)

            # Basic stats
            row[f"ch{ch}_mean"] = float(np.mean(ch_sig))
            row[f"ch{ch}_std"]  = float(np.std(ch_sig))
            row[f"ch{ch}_min"]  = float(np.min(ch_sig))
            row[f"ch{ch}_max"]  = float(np.max(ch_sig))
            row[f"ch{ch}_ptp"]  = float(np.ptp(ch_sig))

            # RMS, kurtosis, skewness
            row[f"ch{ch}_rms"]  = rms(ch_sig)
            row[f"ch{ch}_kurt"] = float(kurtosis(ch_sig, fisher=True, bias=False))
            row[f"ch{ch}_skew"] = float(skew(ch_sig, bias=False))

            # Band powers
            bp = compute_band_powers(ch_sig)
            for band, val in bp.items():
                row[f"ch{ch}_bp_{band}"] = val

            # Spectral entropy
            row[f"ch{ch}_spec_entropy"] = spectral_entropy(ch_sig)

            # HFD
            row[f"ch{ch}_hfd"] = higuchi_fd(ch_sig)

            # MMD
            row[f"ch{ch}_mmd"] = compute_mmd(ch_sig)

            # Zero-crossing
            row[f"ch{ch}_zc"] = zero_crossings(ch_sig)

            # Hjorth params
            activity, mobility, complexity = hjorth_parameters(ch_sig)
            row[f"ch{ch}_hj_activity"] = activity
            row[f"ch{ch}_hj_mobility"] = mobility
            row[f"ch{ch}_hj_complexity"] = complexity

            # Neighbor stats
            mean_n, std_n = neighbor_stats(ch_sig)
            row[f"ch{ch}_neighbor_mean"] = mean_n
            row[f"ch{ch}_neighbor_std"]  = std_n

        rows.append(row)
    return pd.DataFrame(rows)

# --- Add temporal context (t-2, t-1, t+1, t+2) ---
def add_temporal_context(df, group_col="record_id", steps=[1,2]):
    df = df.sort_values(["record_id", "epoch_index"]).reset_index(drop=True)

    # Keep metadata aside
    meta = df[["record_id", "epoch_index"]].copy()

    # Identify feature columns (numerical)
    feature_cols = [c for c in df.columns if c not in ("record_id", "epoch_index")]

    df_feat = df[feature_cols].copy()

    # Generate shifted features
    for step in steps:
        for shift in [-step, step]:
            shifted = df_feat.groupby(meta["record_id"]).shift(shift)
            shifted = shifted.add_suffix(f"_t{shift:+d}")
            df_feat = pd.concat([df_feat, shifted], axis=1)

    # Fill NaNs within each record
    df_feat = df_feat.groupby(meta["record_id"]).ffill().bfill()

    # Fill any remaining NaNs with column medians
    df_feat = df_feat.fillna(df_feat.median())

    # Reattach metadata
    df_out = pd.concat([meta, df_feat], axis=1)

    return df_out




In [15]:
# ==========================================
# BUILD DATASET
# ==========================================
def build_dataset(base_dir, split, drop_invalid=True):
    frames = []
    labels = []
    groups = []

    for rid, sig, tar in iter_signals_and_targets(base_dir, split):
        df = extract_epoch_stats(sig)
        df["record_id"] = rid
        frames.append(df)

        stages = tar
        labels.append(stages[: len(df)])
        groups.append(np.full(len(df), rid))

    features = pd.concat(frames, ignore_index=True)
    y = np.concatenate(labels)
    groups_arr = np.concatenate(groups)

    if drop_invalid:
        mask = np.isin(y, VALID_LABELS)
        features = features.loc[mask].reset_index(drop=True)
        y = y[mask]
        groups_arr = groups_arr[mask]

    feature_cols = [c for c in features.columns if c not in ("record_id", "epoch_index")]
    return features[feature_cols + ["record_id", "epoch_index"]], y, groups_arr, features[["record_id", "epoch_index"]]


In [16]:
# ==========================================
# Build train features
# ==========================================
X_df_full, y, groups, meta = build_dataset(BUNDLE_DIR, "train")
print("Raw train features :", X_df_full.shape)
print("Labels :", y.shape)
print("Unique records :", len(np.unique(groups)))

# Add temporal context (adds shifted columns)
X_df_full = add_temporal_context(X_df_full, group_col="record_id", steps=[1,2])
# drop epoch_index for train features, but keep record_id for later splits
feature_cols = [c for c in X_df_full.columns if c not in ("record_id", "epoch_index")]
X_df = X_df_full[feature_cols].reset_index(drop=True)
meta = X_df_full[["record_id", "epoch_index"]].reset_index(drop=True)

print("After temporal context, features:", X_df.shape)

# Convert to numpy for scaling
scaler = StandardScaler()
X = scaler.fit_transform(X_df)

# Train/val split by groups (records)
splitter = GroupShuffleSplit(test_size=TEST_SIZE, n_splits=1, random_state=RANDOM_STATE)
train_idx, val_idx = next(splitter.split(X, y, groups))

X_train, X_val = X[train_idx], X[val_idx]
y_train, y_val = y[train_idx], y[val_idx]
meta_train, meta_val = meta.iloc[train_idx], meta.iloc[val_idx]

print("Train:", X_train.shape, "Val:", X_val.shape)

Raw train features : (7433, 117)
Labels : (7433,)
Unique records : 8
After temporal context, features: (7433, 1840)
Train: (5802, 1840) Val: (1631, 1840)


In [17]:
# ==========================================
# LIGHTGBM TRAINING (params optimisés)
# ==========================================
lgb_params = {
    "n_estimators": 2000,
    "learning_rate": 0.03,
    "num_leaves": 128,
    "max_depth": -1,
    "min_child_samples": 40,
    "subsample": 0.8,
    "colsample_bytree": 0.8,
    "reg_alpha": 0.0,
    "reg_lambda": 0.0,
    "class_weight": "balanced",
    "random_state": RANDOM_STATE,
    "n_jobs": -1,
}

lgb_model = lgb.LGBMClassifier(**lgb_params)

lgb_model.fit(
    X_train, y_train,
    eval_set=[(X_val, y_val)],
    eval_metric="multi_logloss",
    callbacks=[early_stopping(stopping_rounds=75), log_evaluation(period=50)]
)

# Save model + scaler
joblib.dump({"model": lgb_model, "scaler": scaler, "feature_cols": feature_cols}, MODEL_OUT)
print("Model saved to", MODEL_OUT)

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.042768 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 469200
[LightGBM] [Info] Number of data points in the train set: 5802, number of used features: 1840
[LightGBM] [Info] Start training from score -1.609438
[LightGBM] [Info] Start training from score -1.609438
[LightGBM] [Info] Start training from score -1.609438
[LightGBM] [Info] Start training from score -1.609438
[LightGBM] [Info] Start training from score -1.609438
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
Training until validation scores don't improve for 75 rounds
[L

[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No f

[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[50]	valid_0's multi_logloss: 0.59625
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, bes

In [18]:
# ==========================================
# EVALUATION
# ==========================================
preds_val = lgb_model.predict(X_val)
acc = accuracy_score(y_val, preds_val)
f1 = f1_score(y_val, preds_val, average="macro")
print("Validation ACC:", acc)
print("Validation F1 (macro):", f1)
print(classification_report(y_val, preds_val))

Validation ACC: 0.7982832618025751
Validation F1 (macro): 0.7089598929314256
              precision    recall  f1-score   support

           0       0.91      0.86      0.89       465
           1       0.31      0.57      0.40        76
           2       0.75      0.92      0.82       596
           3       0.97      0.85      0.91       273
           4       0.94      0.37      0.53       221

    accuracy                           0.80      1631
   macro avg       0.78      0.71      0.71      1631
weighted avg       0.84      0.80      0.80      1631



In [19]:
# ==========================================
# BUILD TEST FEATURES & PREDICT
# ==========================================
test_frames = []
test_meta = []

for rid, sig in iter_signals_only(BUNDLE_DIR, "test"):
    df = extract_epoch_stats(sig)
    df["record_id"] = rid
    test_frames.append(df)
    test_meta.append(df[["record_id", "epoch_index"]])

test_full = pd.concat(test_frames, ignore_index=True)
test_meta = pd.concat(test_meta, ignore_index=True)

# Add temporal context to test set (same function)
test_full = add_temporal_context(test_full, group_col="record_id", steps=[1,2])
test_X_df = test_full[feature_cols].reset_index(drop=True)

# Scale with training scaler
test_X = scaler.transform(test_X_df)

# Predict
test_preds = lgb_model.predict(test_X)

# Build identifiers as previously (record_id * 10000 + epoch_index)
identifiers = test_full["record_id"].astype(int) * 10000 + test_full["epoch_index"].astype(int)
submission = pd.DataFrame({
    "identifier": identifiers,
    "target": test_preds.astype(int)
})
submission = submission.sort_values("identifier").reset_index(drop=True)
submission.to_csv(SUBMISSION_OUT, index=False)
print("Submission saved to", SUBMISSION_OUT)
submission.head(20)

Submission saved to /Users/MC/Desktop/CS 3A/Cours/Machine Learning 🖥️/Kaggle Project/beacon-biosignals-sleep-staging-2025/models/lgbm_features_submission.csv


,identifier,target
0,80000,2
1,80001,2
2,80002,2
3,80003,2
4,80004,2
5,80005,2
6,80006,2
7,80007,2
8,80008,2
9,80009,2
